# Sequence Support
Your goal here is to write a program that obtains the support of a list of sequences, in a sequence database.

The program should take as input the following two files:

- FILE1: A list of sequences (one per line) that comprise the dataset. See the file: sequencedb.txt. Lines beginning with '>' are comments, but serve to demarcate one sequence from the next. You may assume that the alphabet will remain Σ={A,C,G,T}

- FILE2: A list of substrings (one per line), i.e., consecutive subsequences, whose supports have to be computed. See the file: seqin.txt.

Your program should output for each sequence in FILE2 the following info:

sequence - support

In [1]:
import numpy as np
from collections import defaultdict

In [2]:
# read file
with open('/content/sequencedb.txt', 'r') as f:
  lines = f.readlines()
  seq_db = [line.strip() for line in lines if not line.startswith('>')]


with open('/content/seqin.txt', 'r') as f:
  lines = f.readlines()
  seq_qry = [line.strip() for line in lines]

In [3]:
alphabet = ['A', 'C', 'G', 'T']
poslist_db = dict()

In [4]:
class SeqPosList:
  def __init__(self, sequence: str) -> None:
    self.sequence = sequence
    self.pos_list = defaultdict(list)

  def sequential_join(self, other: 'SeqPosList') -> 'SeqPosList':
    assert self.sequence[:-1] == other.sequence[:-1], 'Sequence mismatch'

    new_pos_list = defaultdict(list)

    for i in other.pos_list:
      if i in self.pos_list:
        for p in other.pos_list[i]:
          if any(q < p for q in self.pos_list[i]):
            new_pos_list[i].append(p)

    res = SeqPosList(self.sequence + other.sequence[-1])
    res.pos_list = new_pos_list

    return res


In [5]:
for seq_id, seq in enumerate(seq_db):
  for pos_id, char in enumerate(seq):
    if char not in poslist_db:
      poslist_db[char] = SeqPosList(char)  # Initialize for the first time

    poslist_db[char].pos_list[seq_id].append(pos_id)


In [6]:
def compute_support(seq: str) -> SeqPosList:
    """Compute the support of a sequence and insert it into the database"""

    if seq in poslist_db:
        return poslist_db[seq]

    r1 = compute_support(seq[:-1])
    r2 = compute_support(seq[:-2] + seq[-1])

    if len(r1.pos_list.keys()) == 0 or len(r2.pos_list.keys()) == 0:
        poslist_db[seq] = SeqPosList(seq)
        return poslist_db[seq]

    r12 = r1.sequential_join(r2)
    poslist_db[seq] = r12

    return poslist_db[seq]

In [7]:
for seq in seq_qry:
    compute_support(seq)
    print(f'{seq}: {len(poslist_db[seq].pos_list.keys())}')

TAT: 1062
CCAA: 1061
CAGAA: 1061
TCATTT: 1062
GCCG: 1056
GTCAA: 1060
ACGT: 1062
CAAAAAA: 1062
AAAAAA: 1062
AGGCT: 1054
TTTAAACCCGG: 1038
TAGGCA: 1053
